# Embedding-basierte Stellensuche – Übung

**Geschäftliches Problem:** Stellenanzeigen nach ihrer Übereinstimmung mit der Suchanfrage eines Benutzers sortieren.

Ein Modell unterstützt eine Entscheidung, trifft aber nicht die geschäftliche Entscheidung für uns.

## Lernziele

- BM25, für die Informationssuche trainierte Embeddings und die Zusammenführung von Ranglisten vergleichen.
- Relevanzkennzahlen anhand ausdrücklich festgelegter, vorläufiger Relevanzbewertungen berechnen.
- Die Qualität der Suchergebnisse mit den Anforderungen und dem tatsächlichen Nutzen für die Benutzer in Verbindung bringen.

## Datensatz und praktischer Umfang

24 freigegebene, authentische deutsche Textausschnitte. Die Titel wurden anonymisiert und werden bei der Suche nicht berücksichtigt. Für vier Entwicklungs- und acht Testsuchanfragen liegen vom Assistenten erstellte, vorläufige Relevanzbewertungen vor, die noch überprüft werden müssen. T03 wurde nach der Untersuchung früherer Ergebnisse überarbeitet. Es handelt sich deshalb um einen überarbeiteten Vergleich für Unterrichtszwecke und nicht um eine unabhängige Bewertung anhand unangetasteter Testdaten oder um eine Recall-Messung über alle 5'011 vollständigen Stellenanzeigen.

Ergänze die mit # TODO markierten None-Werte, bevor du die betroffenen Codezellen ausführst. Ersetze die Platzhalter für deine Antworten durch jeweils ein bis zwei Sätze.

Abschnitt 7 ergänzt eine interaktive Suche über die **5'011 echten Stellenanzeigen** aus Vorlesung 2. Für diese grössere Sammlung liegen keine Relevanzbewertungen der vollständigen Anzeigen vor. Sie wird deshalb getrennt von der oben beschriebenen Evaluation mit 24 Textausschnitten behandelt. Es handelt sich um unsere ausgewählte Datengruppe und nicht um die gesamte ursprüngliche Datensammlung.

Der Code für die Suche in der grösseren Sammlung ist vollständig vorgegeben. In den früheren Abschnitten der Übung bleiben zwei Programmieraufgaben offen.



## Bibliotheken und Einstellungen

Verwende den vorbereiteten Python-3.11-Kernel im Codespace. Es werden weder eine GPU noch ein API-Schlüssel benötigt.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Minimale Dateninspektion

Es werden ausschliesslich die zitierten Textausschnitte durchsucht. Das ausführlichere Relevanzkriterium beschreibt unsere vorläufig festgelegte Suchabsicht und wird dem Modell nicht als zusätzliche Eingabe übergeben.

In [2]:
import json
import re
from collections import Counter

panel = json.loads((DATA_ROOT / 'nlp_search_benchmark.json').read_text(encoding='utf-8'))
documents = pd.DataFrame([{'id': item['id'], 'title': item['title'],
                           'text': ' [...] '.join(item['fragments'])}
                          for item in panel['documents']])
queries = panel['queries']
display(documents.head(3))
print('Ads:', len(documents), '| Queries:', len(queries), '| Draft judgments:', panel['judgment_status'])

,id,title,text
0,R01,Data pipelines,"Entwicklung, Betrieb und Weiterentwicklung aut..."
1,R02,ETL development,"Du bindest neue Datenquellen an, entwickelst a..."
2,R03,BI analysis,"Erfahrung mit Microsoft Power BI (DAX, Datenmo..."


Ads: 24 | Queries: 12 | Draft judgments: provisional_assistant_draft_pending_instructor_review


## 2. Wortbasiertes Basismodell: BM25

Die bereitgestellte Implementierung verwendet die aus der Vorlesung bekannte Formel, die Worthäufigkeiten, die Seltenheit von Begriffen und die Textlänge berücksichtigt.

Bezeichnungen wie C# und .NET bleiben erhalten.

Stellenanzeigen mit einem Score von 0 werden nicht verwendet, um eine leere Ergebnisliste aufzufüllen. Wenn keine exakten Übereinstimmungen gefunden werden, ist auch eine leere Ergebnisliste ein gültiges Resultat.

In [3]:
def tokens(text):
    return re.findall(r"c\+\+|c#|\.net|[^\W_]+(?:[-'][^\W_]+)*", text.casefold())

def build_bm25(texts):
    counts = [Counter(tokens(text)) for text in texts]
    lengths = np.array([sum(row.values()) for row in counts], dtype=float)
    if not counts or lengths.mean() == 0:
        raise ValueError('BM25 needs a nonempty collection containing searchable words.')
    document_frequency = Counter(term for row in counts for term in row)
    return counts, lengths, document_frequency

def bm25_scores(query, index):
    counts, lengths, document_frequency = index
    scores = np.zeros(len(counts))
    for term in set(tokens(query)):
        frequency = np.array([row[term] for row in counts], dtype=float)
        idf = np.log1p((len(counts) - document_frequency[term] + 0.5)
                      / (document_frequency[term] + 0.5))
        scores += idf * frequency * 2.2 / (
            frequency + 1.2 * (0.25 + 0.75 * lengths / lengths.mean()))
    return scores

def rank_ids(scores, ids, k=3, positive_only=False):
    if len(scores) != len(ids) or not np.isfinite(scores).all():
        raise ValueError('Ranking needs one finite score per document ID.')
    order = sorted(range(len(ids)), key=lambda i: (-scores[i], ids[i]))
    return [ids[i] for i in order if not positive_only or scores[i] > 0][:k]

document_ids = documents['id'].tolist()
small_bm25 = build_bm25(documents['text'])
K, DEPTH = 3, 10
keyword_lists = {query['id']: rank_ids(
    bm25_scores(query['text'], small_bm25), document_ids, DEPTH, True)
    for query in queries}
print('Exact keyword matches for data engineer:', keyword_lists['T03'])

Exact keyword matches for data engineer: ['R03']


## 3. Textpassagen und Suchanfragen codieren

Bei der ersten Verwendung werden die öffentlich verfügbaren E5-Modelldateien heruntergeladen. Die Texte bleiben lokal gespeichert.

Die Präfixe sind modellspezifisch.

Durch normalisierte Vektoren entspricht das Skalarprodukt der Kosinus-Ähnlichkeit.

**Programmieraufgabe:** Berechne alle Ähnlichkeitswerte zwischen Suchanfragen und Stellenanzeigen mit query_vectors @ ad_vectors.T.

In [4]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(2)
MODEL = 'intfloat/multilingual-e5-small'
REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
search_encoder = SentenceTransformer(MODEL, revision=REVISION, device='cpu')
ad_vectors = search_encoder.encode(['passage: ' + text for text in documents['text']],
                                   normalize_embeddings=True, show_progress_bar=False)
query_vectors = search_encoder.encode(['query: ' + query['text'] for query in queries],
                                      normalize_embeddings=True, show_progress_bar=False)
print('Ad-vector shape:', ad_vectors.shape)
scores = query_vectors @ ad_vectors.T  # TODO: query-by-ad cosine score matrix
dense_lists = {query['id']: rank_ids(scores[i], document_ids, DEPTH) for i, query in enumerate(queries)}

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

/home/vscode/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Ad-vector shape: (24, 384)


## 4. Kandidatenlisten kombinieren

**Programmieraufgabe:** Verwende 1 / (60 + position), um den Beitrag jeder Rangliste zu berechnen.

Nicht vorhandene Ergebnisse tragen keine Punkte bei. Ein Ergebnis, das in beiden Listen fehlt, kann durch die Zusammenführung der Ranglisten nicht wiedergefunden werden.

In [5]:
def fuse(rankings, k=3):
    votes = {}
    for ranking in rankings:
        for position, ad in enumerate(ranking, 1):
            contribution = 1 / (60 + position)  # TODO: rank contribution
            votes[ad] = votes.get(ad, 0) + contribution
    return sorted(votes, key=lambda ad: (-votes[ad], ad))[:k]

hybrid_lists = {query['id']: fuse([keyword_lists[query['id']], dense_lists[query['id']]])
                for query in queries}

## 5. Evaluation auf den Entwicklungsdaten

Precision wird durch K geteilt, auch wenn eine Ergebnisliste weniger als K Treffer enthält.

Recall wird durch die Anzahl der vorläufig als relevant bewerteten Textausschnitte geteilt.

Der Reciprocal Rank berücksichtigt die Position des ersten relevanten Ergebnisses. Der Durchschnitt dieser Werte ergibt den Mean Reciprocal Rank (MRR).

Überprüfe den bereitgestellten Code und die Relevanzbewertungen.

In [6]:
def metrics(ranking, relevant, k=3):
    positions = [i for i, ad in enumerate(ranking[:k], 1) if ad in relevant]
    return {'P@3': len(positions) / k,
            'R@3': len(positions) / len(relevant),
            'RR@3': 1 / positions[0] if positions else 0.}

rows = []
recipes = {'BM25': keyword_lists, 'E5': dense_lists, 'RRF': hybrid_lists}
for query in queries:
    for name, lists in recipes.items():
        rows.append({'query': query['id'], 'split': query['split'], 'recipe': name,
                     **metrics(lists[query['id']], set(query['relevant']))})
measurements = pd.DataFrame(rows)
display(measurements[measurements['split'] == 'development']
        .groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean().rename(columns={'RR@3': 'MRR@3'}).round(3))

,P@3,R@3,MRR@3
recipe,,,
BM25,0.417,0.875,0.875
E5,0.500,1.000,1.000
RRF,0.417,0.875,1.000


### Ergebnisse interpretieren

Warum kann ein hoher Ähnlichkeitswert oder eine Übereinstimmung zwischen verschiedenen Suchmethoden trotzdem zu einem ungeeigneten Suchergebnis führen?

**Deine Interpretation:** Ersetze diesen Satz durch deine Antwort.


**Vorgeschlagene Antwort:** Die Scores messen, wie gut die Ergebnisse mit den Bewertungen des Modells beziehungsweise den Suchbegriffen übereinstimmen, nicht aber die unabhängig überprüfte Relevanz. Das entscheidende Kriterium kann im Textausschnitt fehlen, und die Zusammenführung der Ranglisten (Fusion) kann diesen Fehler noch verstärken.

**Selbstkontrolle:** Unterscheide zwischen dem Score, der vorläufigen Relevanzbewertung und den tatsächlichen Hinweisen im Text.

**Schwierigkeitsgrad:** Mittel. Ein häufiger Fehler besteht darin, einen hohen Modell-Score mit einem hohen geschäftlichen Nutzen gleichzusetzen.

## 6. Testvergleich mit festgelegten Einstellungen

Behalte K = 3, eine Kandidatentiefe von 10 und die RRF-Konstante 60 unverändert bei.

Passe diese Einstellungen nach der Betrachtung der Ergebnistabelle nicht erneut an.

Die acht Testsuchanfragen stellen keinen vollständig unabhängigen, unangetasteten Testdatensatz dar.

In [7]:
test = measurements[measurements['split'] == 'test']
display(test.groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean()
        .rename(columns={'RR@3': 'MRR@3'}).round(3))

,P@3,R@3,MRR@3
recipe,,,
BM25,0.417,0.625,0.667
E5,0.583,0.875,0.938
RRF,0.500,0.750,0.938


## Interpretation: Ergebnisse im Detail betrachten, nicht nur den Durchschnitt

Untersuche dieselbe Suchanfrage mit allen drei Methoden.

MATCH bedeutet, dass unser vorläufiges Relevanzkriterium durch den Textausschnitt gestützt wird. Es bedeutet nicht, dass die tatsächliche Eignung der Stellenanzeige überprüft wurde.

In [8]:
selected = next(query for query in queries if query['id'] == 'T03')
print('Query:', selected['text'])
print('Draft criterion:', selected['criterion'])
for name, rankings in recipes.items():
    ids = rankings[selected['id']][:K]
    hits = documents.set_index('id').loc[ids].copy()
    hits['draft match'] = [ad in selected['relevant'] for ad in ids]
    print(name)
    display(hits[['title', 'text', 'draft match']])

Query: data engineer
Draft criterion: For this job-title query, the draft definition of data-engineering work requires evidence of building, developing or operating data pipelines, ETL or ELT for analytics. Querying an existing warehouse without pipeline development is insufficient. An exact job-title match is not required, and anonymized display titles are not indexed.
BM25


,title,text,draft match
id,,,
R03,BI analysis,"Erfahrung mit Microsoft Power BI (DAX, Datenmo...",False


E5


,title,text,draft match
id,,,
R03,BI analysis,"Erfahrung mit Microsoft Power BI (DAX, Datenmo...",False
R01,Data pipelines,"Entwicklung, Betrieb und Weiterentwicklung aut...",True
R04,Power BI reporting,Sehr gute Kenntnisse in Power BI (DAX) und Exc...,False


RRF


,title,text,draft match
id,,,
R03,BI analysis,"Erfahrung mit Microsoft Power BI (DAX, Datenmo...",False
R01,Data pipelines,"Entwicklung, Betrieb und Weiterentwicklung aut...",True
R04,Power BI reporting,Sehr gute Kenntnisse in Power BI (DAX) und Exc...,False


### Ergebnisse interpretieren

Welcher Ansatz sollte weiter untersucht werden? Was müsste eine Evaluation vor dem praktischen Einsatz zusätzlich berücksichtigen?

**Deine Interpretation:** Ersetze diesen Satz durch deine Antwort.

## 7. Die grössere Sammlung echter Stellenanzeigen durchsuchen

Wende nun dieselben drei Methoden auf **alle 5'011 echten Stellenanzeigen aus Vorlesung 2** an.

Die freigegebene komprimierte Datei ist bereits vorhanden. Es müssen keine neuen Daten heruntergeladen werden.

Im Gegensatz zu den 24 kurzen Textausschnitten handelt es sich hier um vollständige, bereinigte Stellenbeschreibungen.

Wir kombinieren die zeitlich getrennten Datensätze aus Vorlesung 2 zu einem einzigen durchsuchbaren Datenbestand.

Es wird kein neues Modell zur Kategorisierung trainiert: E5 bleibt unverändert, und die Kategorien werden lediglich neben den Suchergebnissen angezeigt.

Damit wird nicht bewertet, wie gut die Suche bei zukünftigen Stellenanzeigen funktionieren würde.

Die frühere Evaluation behält ihre eigenen Dokumente, Suchanfragen und Relevanzbewertungen. Ihre IDs und Bewertungen dürfen **nicht** auf die Ergebnisse der grösseren Sammlung übertragen werden.

Hier untersuchen wir die Suchergebnisse, anstatt Precision oder Recall zu berechnen.

In [9]:
LARGE_DATA_PATH = DATA_ROOT / 'nlp_real_job_categories.csv.gz'
large_documents = pd.read_csv(LARGE_DATA_PATH)
required = ['teaching_id', 'text', 'category']
if not set(required).issubset(large_documents.columns):
    raise ValueError('The real-ad file needs teaching_id, text and category columns.')
if large_documents[required].isna().any().any():
    raise ValueError('The real-ad collection contains missing IDs, text or categories.')
if (not large_documents['teaching_id'].is_unique
        or not large_documents['text'].map(lambda text: isinstance(text, str) and bool(text.strip())).all()):
    raise ValueError('The real-ad collection needs unique IDs and nonempty descriptions.')
large_documents = large_documents.rename(columns={'teaching_id': 'id'})
large_ids = large_documents['id'].tolist()
large_bm25 = build_bm25(large_documents['text'])
print(f'Indexed {len(large_documents):,} real ads for keyword search.')
print('Category labels are displayed for orientation, not used to rank ads.')

Indexed 5,011 real ads for keyword search.
Category labels are displayed for orientation, not used to rank ads.


### Die Vektoren der Stellenanzeigen einmalig erstellen

Der zeitaufwendige Schritt besteht darin, jede Stellenbeschreibung mit E5 zu verarbeiten.

Wir verarbeiten die Anzeigen in kleinen Gruppen (Batches) und zeigen einen Fortschrittsbalken an.

Anschliessend speichern wir die Vektoren in einem lokalen **Cache**. Das ist eine Datei, mit der wir die bereits durchgeführte Berechnung bei der nächsten Verwendung wiederverwenden können.

Der Code überprüft vor der Wiederverwendung, ob die gespeicherten Vektoren zu den Daten, dem Modell und den Einstellungen passen.

Wenn sich die Einstellungen geändert haben, werden die Vektoren neu erstellt und dies wird deutlich angezeigt.

Ein beschädigter Cache führt zu einer Fehlermeldung. Lösche bei Bedarf 00_Data/nlp-search-cache/real-job-e5-vectors.npz und führe diese Codezelle erneut aus.

Der Cache wird von Git ignoriert. Veröffentliche ihn nicht unter der Annahme, dass es sich um anonymisierte Daten handelt.

**Wichtiger Unterschied bei den Eingabedaten:** BM25 durchsucht die vollständige, bereinigte Stellenbeschreibung. E5 verarbeitet dagegen nur den Anfang des Textes bis zu seiner Token-Grenze.

Der Code zeigt an, wie viele Stellenanzeigen diese Grenze überschreiten.

Eine Anforderung am Ende einer Stellenanzeige kann deshalb von BM25 berücksichtigt werden, während sie für E5 unsichtbar bleibt.

Der Vergleich untersucht somit nicht ausschliesslich die Unterschiede zwischen den Modellarchitekturen unter identischen Bedingungen.

Eine mögliche spätere Erweiterung besteht darin, lange Stellenanzeigen in mehrere Textabschnitte aufzuteilen.

Es werden keine Vektoren anhand der Stellenanzeigen trainiert. Das bereits vorhandene E5-Modell wandelt die Texte in Zahlen um. Der Cache speichert diese Zahlen und keine neu trainierten Modellgewichte.

Bereite diesen Schritt vor der Demonstration in der Vorlesung vor.

In [ ]:
import hashlib
import time
from importlib.metadata import version

CACHE_PATH = DATA_ROOT / 'nlp-search-cache' / 'real-job-e5-vectors.npz'
LARGE_BATCH_SIZE = 16
passages = ['passage: ' + text for text in large_documents['text']]
input_lengths = np.array([
    len(search_encoder.tokenizer.encode(text, truncation=False, verbose=False))
    for text in passages
])
input_limit = int(search_encoder.max_seq_length)
print(f'E5 input limit: {input_limit} tokens, including the passage prefix and special markers.')
print(f'Ads exceeding that limit: {int((input_lengths > input_limit).sum()):,}'
      f' / {len(passages):,}. Their endings are not encoded.')

cache_settings = {
    'format': 1,
    'data_sha256': hashlib.sha256(LARGE_DATA_PATH.read_bytes()).hexdigest(),
    'model': MODEL, 'revision': REVISION,
    'input_limit': input_limit, 'passage_prefix': 'passage: ',
    'normalized': True, 'batch_size': LARGE_BATCH_SIZE,
    'dimensions': int(ad_vectors.shape[1]),
    'versions': {package: version(package) for package in
                 ['sentence-transformers', 'transformers', 'torch', 'numpy']},
}

def validate_cached_vectors(vectors, row_count, dimensions):
    if vectors.shape != (row_count, dimensions) or vectors.dtype != np.float32:
        raise ValueError('Unexpected cached vector shape or type. Delete the cache file and rerun.')
    if not np.isfinite(vectors).all() or not np.allclose(
            np.linalg.norm(vectors, axis=1), 1, atol=1e-4):
        raise ValueError('Invalid cached vectors. Delete the cache file and rerun.')

large_vectors = None
if CACHE_PATH.exists():
    with np.load(CACHE_PATH, allow_pickle=False) as saved:
        stored_settings = json.loads(saved['settings'].item())
        if stored_settings == cache_settings:
            large_vectors = saved['vectors']
    if large_vectors is None:
        print('The data, model or settings changed; rebuilding the cached vectors.')
    else:
        validate_cached_vectors(large_vectors, len(large_documents), cache_settings['dimensions'])
        print('Loaded matching cached E5 vectors; no ad encoding needed.')

if large_vectors is None:
    print('Creating E5 vectors. The progress bar shows the batches completed.', flush=True)
    started = time.perf_counter()
    large_vectors = search_encoder.encode(
        passages, batch_size=LARGE_BATCH_SIZE, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True).astype(np.float32, copy=False)
    validate_cached_vectors(large_vectors, len(large_documents), cache_settings['dimensions'])
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = CACHE_PATH.with_suffix('.tmp')
    with temporary_path.open('wb') as temporary:
        np.savez_compressed(temporary, vectors=large_vectors,
                            settings=json.dumps(cache_settings, sort_keys=True))
    temporary_path.replace(CACHE_PATH)
    print(f'Encoding finished in {time.perf_counter() - started:.1f} seconds; vectors saved locally.')

print('Vector matrix:', large_vectors.shape,
      f'| Memory: {large_vectors.nbytes / 1024**2:.2f} MiB')

E5 input limit: 512 tokens, including the passage prefix and special markers.
Ads exceeding that limit: 2,498 / 5,011. Their endings are not encoded.
Creating E5 vectors. The progress bar shows the batches completed.


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

### Probiere deine eigene Suchanfrage aus

Ändere user_query in der folgenden Codezelle und führe anschliessend **nur diese Codezelle** erneut aus, um eine neue Suche durchzuführen.

Die Stellenanzeigen müssen dafür nicht erneut codiert werden.

Jede Methode zeigt bis zu fünf Vorschläge an.

Die hybride Suche kombiniert die ersten 20 Kandidaten jeder Methode mit derselben Reciprocal-Rank-Fusion-Regel wie zuvor.

BM25 kann weniger Ergebnisse oder auch gar keine Treffer liefern. E5 findet dagegen weiterhin die Vektoren mit der grössten Ähnlichkeit, selbst wenn keiner davon tatsächlich gut zur Suchanfrage passt.

Lies die Stellenbeschreibungen und achte nicht nur auf die Rangpositionen.

Die angezeigte Vorschau enthält lediglich den Anfang jeder Stellenanzeige und kann wichtige Informationen auslassen.

Um lokal eine vollständige Stellenbeschreibung anzusehen, verwende print(large_lookup.loc['JOB-00001', 'text']) und ersetze die ID durch eine zurückgegebene Teaching-ID.

Kopiere keine zusätzlichen Stellenbeschreibungen in öffentlich geteilte Antworten.

**Hier gibt es keine automatisch berechneten Qualitätskennzahlen.** Die Relevanzbewertungen der 24 Textausschnitte decken diese Sammlung nicht ab.

Nicht bewertete Stellenanzeigen sind nicht automatisch irrelevant. Ausserdem sind Stellenkategorien keine Relevanzbewertungen für Suchanfragen.

Um Precision@K zu berechnen, müssten die zurückgegebenen Stellenanzeigen hinsichtlich ihrer Relevanz bewertet werden.

Für Recall@K müsste zusätzlich bekannt sein, welche anderen Stellenanzeigen in der Sammlung relevant sind.

In [ ]:
user_query = 'Data Engineer mit Python und SQL'
LARGE_K, LARGE_DEPTH = 5, 20

if not isinstance(user_query, str) or not user_query.strip():
    raise ValueError('Enter a nonempty search query.')
if not 1 <= LARGE_K <= LARGE_DEPTH <= len(large_documents):
    raise ValueError('Use 1 <= LARGE_K <= LARGE_DEPTH <= the number of indexed ads.')
large_keyword_scores = bm25_scores(user_query, large_bm25)
query_input = 'query: ' + user_query.strip()
query_length = len(search_encoder.tokenizer.encode(query_input, truncation=False, verbose=False))
if query_length > search_encoder.max_seq_length:
    raise ValueError('This query exceeds the E5 input limit; shorten it before comparing results.')
large_query_vector = search_encoder.encode(
    [query_input], normalize_embeddings=True, show_progress_bar=False)
large_dense_scores = (large_query_vector @ large_vectors.T)[0]
large_keyword_ids = rank_ids(large_keyword_scores, large_ids, LARGE_DEPTH, True)
large_dense_ids = rank_ids(large_dense_scores, large_ids, LARGE_DEPTH)
large_rankings = {
    'BM25': large_keyword_ids[:LARGE_K],
    'E5': large_dense_ids[:LARGE_K],
    'Hybrid (RRF)': fuse([large_keyword_ids, large_dense_ids], k=LARGE_K),
}
print('Query:', user_query)
print('No relevance metrics: these full-ad results have not been judged.')
large_lookup = large_documents.set_index('id')
for name, ranking in large_rankings.items():
    print(f'\n{name}: {len(ranking)} suggestions')
    if not ranking:
        print('No keyword matches. The list is deliberately left empty.')
        continue
    hits = large_lookup.loc[ranking, ['category', 'text']].copy()
    hits.insert(0, 'rank', np.arange(1, len(hits) + 1))
    hits['description preview'] = hits.pop('text').str.slice(0, 260)
    display(hits)

### Die Suchergebnisse der grösseren Sammlung interpretieren

- Erkläre für eine Suchanfrage einen Unterschied zwischen den Ergebnislisten von BM25, E5 und der hybriden Suche. Verwende dazu konkrete Informationen aus den Stellenbeschreibungen.

- Welche zwingend erforderliche Bedingung würdest du überprüfen, bevor du einer stellensuchenden Person ein Ergebnis anzeigst?

- Warum können wir die zuvor berechneten Recall-Werte nicht für diese Sammlung wiederverwenden?

**Deine Interpretation:** Ersetze diesen Satz durch deine Antwort.

**Hinweise zur Beantwortung:** Nenne eine tatsächlich zurückgegebene Anzeigen-ID und die Wörter im Text, die die Suchanfrage unterstützen oder ihr widersprechen. BM25 kann sich auf exakte Suchbegriffe stützen; E5 kann auch verwandte Formulierungen finden. Eine hohe Ähnlichkeit allein beweist jedoch nicht, dass eine Anzeige tatsächlich passt. Die hybride Suche kann ein schlechtes Ergebnis zusätzlich verstärken oder eine Anzeige übersehen, die in keiner der beiden Kandidatenlisten enthalten ist. Überprüfe zwingende Anforderungen wie Arbeitsort, Qualifikationen oder Schichtzeiten anhand der vollständigen Stellenbeschreibung. Durch die Kürzung zu langer Texte bei E5 können relevante Informationen verborgen bleiben.

**Bewertungskriterien:** Ein Vergleich, der sich auf konkrete Textbelege stützt, eine konkrete Anforderung, die überprüft werden muss, und eine korrekte Erklärung, dass sich die bisherigen Relevanzbewertungen auf andere Texte und eine wesentlich kleinere Sammlung beziehen.

**Häufige Fehler:** Anzunehmen, dass das höchstplatzierte Suchergebnis automatisch geeignet ist, eine Kategoriebezeichnung als Beweis für Relevanz zu betrachten oder jede nicht bewertete Stellenanzeige als nicht relevanten Treffer einzustufen.

**Schwierigkeitsgrad:** Mittel.

## Zusammenfassung und geschäftliche Auswirkungen

Eine gute Relevanzbewertung anhand vorhandener Testdaten bedeutet nicht automatisch einen hohen Nutzen für die Benutzer im praktischen Einsatz.

Wenn ein Textausschnitt keine Hinweise auf die Relevanz enthält, beweist das nicht, dass die gesamte Stellenanzeige irrelevant ist.

Verändere Relevanzbewertungen nicht nachträglich und unbemerkt, nur damit ein bestimmtes Modell besser abschneidet.

## Einschränkungen

Dies ist ein kleines Lehrbeispiel und kein Nachweis für die Leistungsfähigkeit im praktischen Einsatz.

Halte die Evaluation von der Modellanpassung und Modellauswahl getrennt.

Untersuche Fehler und die Entscheidungen der Benutzer, statt dich nur auf eine einzelne Kennzahl zu konzentrieren.

Die grössere Suche verwendet 5'011 echte Stellenanzeigen, verfügt aber über keine überprüften Relevanzbewertungen für die vollständigen Anzeigen.

BM25 berücksichtigt die vollständigen bereinigten Beschreibungen, während E5 lange Eingabetexte kürzt.

Interpretiere weder die grössere Auswahl an Kandidaten noch einen hohen Ähnlichkeitswert als Beweis für eine bessere Suchqualität.

Das Entfernen personenbezogener Informationen und die Verwendung von Embeddings garantieren keine Anonymität.

## Weiterführende Übungen

- Untersuche die Suchanfrage zu C# und .NET (T07) sowie die Suchanfrage zu Arbeitsschichten (T08).
- Kann sich Recall verändern, obwohl die Position des ersten relevanten Ergebnisses gleich bleibt?
- Optional: Vergleiche gewöhnliches BERT mit Pooling, die Aufteilung langer Texte in Abschnitte (Chunking) oder ein erneutes Ranking (Reranking) anhand neuer, überprüfter Daten.
- Wähle eine neue Suchanfrage für die grössere Sammlung, bewerte die zusammengeführten besten Ergebnisse aller drei Methoden und kennzeichne nicht bewertete Ergebnisse ausdrücklich. Behaupte auf Grundlage dieser begrenzten Bewertung nicht, den Recall für die gesamte Sammlung bestimmt zu haben.

## References

**Required reading:** https://www.sbert.net/examples/applications/semantic-search/README.html
and https://huggingface.co/intfloat/multilingual-e5-small
**Instructor references:** Reimers & Gurevych (2019), *Sentence-BERT*, https://aclanthology.org/D19-1410/;
Wang et al. (2024), *Multilingual E5 Text Embeddings*, https://arxiv.org/abs/2402.05672;
Cormack et al. (2009), *Reciprocal rank fusion*, https://doi.org/10.1145/1571941.1572114.
**Official BM25 documentation:** https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.